In [1]:
#import
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import requests
import os
from gwosc.datasets import find_datasets
from gwosc import datasets
import time

Matplotlib is building the font cache; this may take a moment.


In [6]:
#getting all GWTC events
url = "https://gwosc.org/api/v2/catalogs/GWTC/events"
params = {"include-default-parameters": "true", "pagesize": 20}

all_events = []

while True:
    #Send a GET request to GWOSC with page number as a query parameter
    response = requests.get(url, params)
    #for page 1, this is the same thing as: https://gwosc.org/api/v2/catalogs/GWTC/events?include-default-parameters=true&pagesize=20&page=1
    
    # Parse the JSON body into a Python dictionary
    data = response.json()

    #grab event data, which is under "results," and add to running list
    all_events.extend(data["results"])

    # If there's no next page, stop looping
    if data["next"] is None:
        break

    # Otherwise advance to the next page
    url = data["next"] #next page url is under "next"
    params = {} #clearing because next url has all the parameters
    time.sleep(0.5)  #wait half a second between requests to not overload the server

print(f"\nTotal events fetched: {len(all_events)}") #for debugging


Total events fetched: 391


In [8]:
#filtering for BBH events
def is_bbh(event):
    params = {p["name"]: p["best"] for p in event.get("default_parameters", [])} #return empty list if "default_parameters" is somehow missing
    #get the list under dictionary key "default paraemeters" in event, and we are traversing each item p in the list, which are dictionaries, and make take p["name"] the keys and p["best"] as values for this new dictionary params
    mass_one = params.get("mass_1_source", 0)  # returns 0 if missing, won't crash
    mass_two = params.get("mass_2_source", 0)  # returns 0 if missing, won't crash
    return mass_one > 3 and mass_two > 3

bbh_events = [e for e in all_events if is_bbh(e)] #filter
print(f"Total events: {len(all_events)}") #for debugging
print(f"BBH events: {len(bbh_events)}") #for debugging
for e in bbh_events[:5]:
    print(e["name"]) #for debugging

Total events: 391
BBH events: 273
GW250119_190238
GW250119_025138
GW250118_170523
GW250118_055802
GW250118_023225


In [13]:
#get skymap url for each event. I am doing "GWTC-1-confident", "GWTC-2.1-confident", "GWTC-3-confident", "GWTC-4.0" because only these events have skymaps

url = "https://gwosc.org/api/v2/event-versions?release=GWTC-1-confident%2CGWTC-2.1-confident%2CGWTC-3-confident%2CGWTC-4.0&lastver=true&include-default-parameters=true&format=json"
#this is the url for GWTC-4.0 cumulative. Replace this once GWTC-5.0 has skymaps

bbh_with_skymaps = []

while True:
    #GET request
    response = requests.get(url, {})
    # Parse the JSON body into a Python dictionary
    data = response.json()
    for event in data["results"]: #results is a dictionary
        if is_bbh(event):
            bbh_with_skymaps.append(event)
    if data["next"] is None:
        break
    url = data["next"]
    time.sleep(0.5)

print(f"Total bbh events with skymaps: {len(bbh_with_skymaps)}")

Total bbh events with skymaps: 167


In [25]:
#Get skymap URLS from bbh_with_skymaps
# GWTC-1: individual .fits.gz files on DCC, one per event
# GWTC-2.1 and GWTC-3: skymap URL is in the API under "label": "skymap" on the preferred result
# GWTC-4.0: single tar.gz on Zenodo containing all events

GWTC1_SKYMAP_URL = "https://dcc.ligo.org/public/0157/P1800381/007/{name}_skymap.fits.gz"
GWTC4_SKYMAP_TAR = "https://zenodo.org/records/17602505/files/IGWN-GWTC4p0-38214bd95_724-Archived_Skymaps.tar.gz?download=1"

skymap_urls = {}  # name -> url or tar info

for event in bbh_with_skymaps:
    name = event["name"]
    catalog = event["catalog"]

    if catalog == "GWTC-1-confident":
        skymap_urls[name] = GWTC1_SKYMAP_URL.format(name=name)
    elif catalog in ("GWTC-2.1-confident","GWTC-3-confident"):
        short_name = event["shortName"] 
        parameters_url = f"https://gwosc.org/api/v2/event-versions/{short_name}/parameters?format=json"

        response = requests.get(parameters_url, {})
        data = response.json()

        for result in data["results"]:
            if result["is_preferred"]:
                for link in result["links"]:
                    if link["label"] == "skymap":
                        skymap_urls[name] = link["url"]
        
        time.sleep(0.3)

    elif catalog == "GWTC-4.0":
        skymap_urls[name] = GWTC4_SKYMAP_TAR

print(f"Skymap URLs found: {len(skymap_urls)} / {len(bbh_with_skymaps)}")
for name, url in list(skymap_urls.items())[:5]:
    print(f"  {name}: {url[:80]}...")


Skymap URLs found: 167 / 167
  GW240109_050431: https://zenodo.org/records/17602505/files/IGWN-GWTC4p0-38214bd95_724-Archived_Sk...
  GW240107_013215: https://zenodo.org/records/17602505/files/IGWN-GWTC4p0-38214bd95_724-Archived_Sk...
  GW240104_164932: https://zenodo.org/records/17602505/files/IGWN-GWTC4p0-38214bd95_724-Archived_Sk...
  GW231231_154016: https://zenodo.org/records/17602505/files/IGWN-GWTC4p0-38214bd95_724-Archived_Sk...
  GW231230_170116: https://zenodo.org/records/17602505/files/IGWN-GWTC4p0-38214bd95_724-Archived_Sk...
